# Import libraries

In [36]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
import joblib


# Load dataset

In [2]:
df = pd.read_csv(r"C:\Users\name\Desktop\BIG FOLDER\PROJECTS\My_Projects\bank_deposit_prediction\data\bank.csv")

I will drop the variable "day" because it doesn't show the exact days of the week which in that sense makes it less valuable for our prediction. Duration is a valid predictor and will be retained bacause the project is a post-contact subscription analysis, where campaign interaction variables such as contact duration, campaign frequency, and previous campaign history are available at prediction time. 

In [3]:
df1 = df.copy()
df1.head()

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,deposit
0,59,admin.,married,secondary,no,2343,yes,no,unknown,5,may,1042,1,-1,0,unknown,yes
1,56,admin.,married,secondary,no,45,no,no,unknown,5,may,1467,1,-1,0,unknown,yes
2,41,technician,married,secondary,no,1270,yes,no,unknown,5,may,1389,1,-1,0,unknown,yes
3,55,services,married,secondary,no,2476,yes,no,unknown,5,may,579,1,-1,0,unknown,yes
4,54,admin.,married,tertiary,no,184,no,no,unknown,5,may,673,2,-1,0,unknown,yes


In [4]:
df1.drop(columns='day', axis=0,inplace=True)
df1.drop(columns='pdays', axis=0,inplace=True)

In [5]:
df1

,age,job,marital,education,default,balance,housing,loan,contact,month,duration,campaign,previous,poutcome,deposit
0,59,admin.,married,secondary,no,2343,yes,no,unknown,may,1042,1,0,unknown,yes
1,56,admin.,married,secondary,no,45,no,no,unknown,may,1467,1,0,unknown,yes
2,41,technician,married,secondary,no,1270,yes,no,unknown,may,1389,1,0,unknown,yes
3,55,services,married,secondary,no,2476,yes,no,unknown,may,579,1,0,unknown,yes
4,54,admin.,married,tertiary,no,184,no,no,unknown,may,673,2,0,unknown,yes
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11157,33,blue-collar,single,primary,no,1,yes,no,cellular,apr,257,1,0,unknown,no
11158,39,services,married,secondary,no,733,no,no,unknown,jun,83,4,0,unknown,no
11159,32,technician,single,secondary,no,29,no,no,cellular,aug,156,2,0,unknown,no
11160,43,technician,married,secondary,no,0,no,yes,cellular,may,9,2,5,failure,no


In [6]:
df1.to_csv(r"C:\Users\name\Desktop\BIG FOLDER\PROJECTS\My_Projects\bank_deposit_prediction\data\cleaned_bank.csv")

# Encode target

In [7]:
df1["deposit"] = df1["deposit"].map({"no": 0, "yes": 1})

# Confirm
print(df["deposit"].value_counts())

deposit
no     5873
yes    5289
Name: count, dtype: int64


# Separate features and target

In [8]:
X = df1.drop(columns=["deposit"])
y = df1["deposit"]

In [9]:
X

,age,job,marital,education,default,balance,housing,loan,contact,month,duration,campaign,previous,poutcome
0,59,admin.,married,secondary,no,2343,yes,no,unknown,may,1042,1,0,unknown
1,56,admin.,married,secondary,no,45,no,no,unknown,may,1467,1,0,unknown
2,41,technician,married,secondary,no,1270,yes,no,unknown,may,1389,1,0,unknown
3,55,services,married,secondary,no,2476,yes,no,unknown,may,579,1,0,unknown
4,54,admin.,married,tertiary,no,184,no,no,unknown,may,673,2,0,unknown
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11157,33,blue-collar,single,primary,no,1,yes,no,cellular,apr,257,1,0,unknown
11158,39,services,married,secondary,no,733,no,no,unknown,jun,83,4,0,unknown
11159,32,technician,single,secondary,no,29,no,no,cellular,aug,156,2,0,unknown
11160,43,technician,married,secondary,no,0,no,yes,cellular,may,9,2,5,failure


# Encode Categorical Variables

In [10]:
# Separate column types
categorical_cols = X.select_dtypes(include=["object"]).columns.tolist()
numerical_cols = X.select_dtypes(exclude=["object"]).columns.tolist()

print("Categorical columns:", categorical_cols)
print("Numerical columns:", numerical_cols)

Categorical columns: ['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'poutcome']
Numerical columns: ['age', 'balance', 'duration', 'campaign', 'previous']


# Train test split

In [11]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [12]:
print(X_train.shape, X_test.shape)
print(y_train.shape, y_test.shape)

(8929, 14) (2233, 14)
(8929,) (2233,)


save the test data for future use

In [13]:
test_data = pd.concat([X_test, y_test], axis=1)
test_data.to_csv(r"C:\Users\name\Desktop\BIG FOLDER\PROJECTS\My_Projects\bank_deposit_prediction\data\test_data.csv",index=False)

# Build a preprocessing pipeline

In [14]:
# Preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numerical_cols),
        ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), categorical_cols)
    ]
)

# Logistic pipeline

In [15]:
# Full pipeline
log_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000))
])
# Fit on training data
log_model.fit(X_train, y_train)

,steps,"[('preprocessor', ...), ('classifier', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


# Save the model

In [16]:
joblib.dump(log_model, r"C:\Users\name\Desktop\BIG FOLDER\PROJECTS\My_Projects\bank_deposit_prediction\model\log_model.pkl")

['C:\\Users\\name\\Desktop\\BIG FOLDER\\PROJECTS\\My_Projects\\bank_deposit_prediction\\model\\log_model.pkl']

# SVC pipeline

In [51]:
svc_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", SVC(kernel="rbf", C=1.0, gamma="scale", probability=True, random_state=42))
])

# Train
svc_model.fit(X_train, y_train)

,steps,"[('preprocessor', ...), ('classifier', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [18]:
print("Preprocessing and model training complete.")

Preprocessing and model training complete.


# Save the model

In [52]:
# Save the entire pipeline (preprocessing + model)
joblib.dump(svc_model, r"C:\Users\name\Desktop\BIG FOLDER\PROJECTS\My_Projects\bank_deposit_prediction\model\svc__model.pkl")


['C:\\Users\\name\\Desktop\\BIG FOLDER\\PROJECTS\\My_Projects\\bank_deposit_prediction\\model\\svc__model.pkl']

# KNN pipeline

In [47]:
knn_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", KNeighborsClassifier(n_neighbors=9))
])

# Train
knn_model.fit(X_train, y_train)

,steps,"[('preprocessor', ...), ('classifier', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


## Save the model

In [48]:
joblib.dump(knn_model, r"C:\Users\name\Desktop\BIG FOLDER\PROJECTS\My_Projects\bank_deposit_prediction\model\knn__model.pkl")

['C:\\Users\\name\\Desktop\\BIG FOLDER\\PROJECTS\\My_Projects\\bank_deposit_prediction\\model\\knn__model.pkl']

# Decision Tree pipeline

In [ ]:
dt_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", DecisionTreeClassifier(random_state=42))
])

# Train
dt_model.fit(X_train, y_train)

,steps,"[('preprocessor', ...), ('classifier', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


## Save the model

In [33]:
joblib.dump(dt_model, r"C:\Users\name\Desktop\BIG FOLDER\PROJECTS\My_Projects\bank_deposit_prediction\model\dt__model.pkl")

['C:\\Users\\name\\Desktop\\BIG FOLDER\\PROJECTS\\My_Projects\\bank_deposit_prediction\\model\\dt__model.pkl']

# Random Forest pipeline

In [49]:
# Preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        ("num", "passthrough", numerical_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols)
    ]
)

rf_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=200,
        max_depth=None,
        min_samples_split=2,
        min_samples_leaf=1,
        random_state=42,
        n_jobs=-1
    ))
])
# Train
rf_model.fit(X_train, y_train)

,steps,"[('preprocessor', ...), ('classifier', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [50]:
joblib.dump(rf_model, r"C:\Users\name\Desktop\BIG FOLDER\PROJECTS\My_Projects\bank_deposit_prediction\model\rf__model.pkl")

['C:\\Users\\name\\Desktop\\BIG FOLDER\\PROJECTS\\My_Projects\\bank_deposit_prediction\\model\\rf__model.pkl']